# Spark server lay of the land

This notebook is a quick survey of the Spark environment available to a notebook session. The goal is not to run a full workload yet; it is to answer practical questions first:

- What Spark session and master am I connected to?
- Which executors are running tasks?
- What does each executor's filesystem look like?
- Which common paths exist and are writable?
- Is a file written by one task visible to tasks running elsewhere?

Those checks are useful before pointing larger analyses at the cluster, especially when deciding where input data, temporary files, caches, and outputs should live.

## Connect to Spark

Fill in the cluster-specific imports and session setup here. The remaining cells assume there is a `SparkSession` named `spark` and a `SparkContext` named `sc`.

For a local smoke test, a minimal session such as `SparkSession.builder.master("local[*]").getOrCreate()` is usually enough. On AstroFlow or another managed deployment, replace that with the project-specific setup code.

In [ ]:
# custom setup modules
from spark_setup import SparkSetup
from setup_data_gaia_dr3 import SetupDataGaiaDR3
from astroflow_spark_gaia import spark

sc = spark.sparkContext

## Summarize the driver and Spark session

Start with the driver-side view of the application. The Spark version, master URL, application id, and default parallelism are the first sanity checks that the notebook is attached to the expected environment.

In [ ]:
from pprint import pprint

print("Spark version:", spark.version)
print("Spark master:", sc.master)
print("Application id:", sc.applicationId)
print("Default parallelism:", sc.defaultParallelism)

try:
    executor_memory_status = sc._jsc.sc().getExecutorMemoryStatus()
    print("Executor memory status:")
    print(executor_memory_status.toString())
except Exception as exc:
    print("Could not read executor memory status from the JVM SparkContext:", exc)

## Inspect executor task environments

Spark does not have a direct equivalent to Dask's `Client.run`. To inspect executors, run a small job and collect one record from each partition. This reveals which hosts and Python worker processes are actually executing tasks.

In [ ]:
def executor_env_info(_):
    import os
    import platform
    import socket
    import sys

    from pyspark import TaskContext

    context = TaskContext.get()
    yield {
        "partition_id": context.partitionId() if context else None,
        "attempt_number": context.attemptNumber() if context else None,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "cwd": os.getcwd(),
        "python": sys.executable,
        "python_version": sys.version,
        "platform": platform.platform(),
        "pyspark_python": os.environ.get("PYSPARK_PYTHON", ""),
    }


num_tasks = max(sc.defaultParallelism, 2)
executor_envs = sc.parallelize(range(num_tasks), num_tasks).mapPartitions(executor_env_info).collect()
pprint(executor_envs)

## Inspect executor filesystems

Listing `/` from executor tasks gives a quick map of the container or host environment each task can see. Look for expected mount points such as `/mnt`, `/scratch`, `/data`, or project-specific directories. If those paths are absent on executors, Spark tasks cannot read data from them even if the notebook driver can.

In [ ]:
def check_executor_fs(_):
    import os
    import socket

    from pyspark import TaskContext

    context = TaskContext.get()
    yield {
        "partition_id": context.partitionId() if context else None,
        "hostname": socket.gethostname(),
        "root_entries": sorted(os.listdir("/")),
    }


executor_filesystems = sc.parallelize(range(num_tasks), num_tasks).mapPartitions(check_executor_fs).collect()
pprint(executor_filesystems)

## Check candidate working directories

This check separates three cases for each path: `True` means the path exists and is writable by the executor process, `False` means it exists but is not writable, and `None` means the path is not present for that task. A good shared working directory should appear consistently across executor tasks, not just on the driver.

In [ ]:
def check_writable_paths(_):
    import os
    import socket

    from pyspark import TaskContext

    candidates = ["/mnt", "/tmp", "/home", "/scratch", "/data"]
    context = TaskContext.get()
    results = {}
    for path in candidates:
        if os.path.exists(path):
            results[path] = os.access(path, os.W_OK)
        else:
            results[path] = None

    yield {
        "partition_id": context.partitionId() if context else None,
        "hostname": socket.gethostname(),
        "writable": results,
    }


writable_paths = sc.parallelize(range(num_tasks), num_tasks).mapPartitions(check_writable_paths).collect()
pprint(writable_paths)

## Test whether a path is shared

Writable is not the same as shared. The next two cells write a marker file from one Spark task and then try to read it from many tasks. If only tasks on the same host can see the file, the path is executor-local. If tasks across hosts can see the same contents, the path is backed by shared storage.

Spark does not guarantee that a given partition runs on a specific executor, so treat this as a practical smoke test rather than a complete storage audit.

In [ ]:
test_path = "/tmp/spark_shared_test.txt"  # adjust based on the writable-path results


def write_test_file(_):
    import os
    import socket

    try:
        with open(test_path, "w") as handle:
            handle.write(f"hello from {socket.gethostname()} pid {os.getpid()}")
        status = f"wrote to {test_path}"
    except Exception as exc:
        status = f"failed: {exc}"

    yield {
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "status": status,
    }


write_result = sc.parallelize([0], 1).mapPartitions(write_test_file).collect()
pprint(write_result)

In [ ]:
def read_test_file(_):
    import os
    import socket

    from pyspark import TaskContext

    context = TaskContext.get()
    exists = os.path.exists(test_path)
    contents = None
    if exists:
        with open(test_path) as handle:
            contents = handle.read()

    yield {
        "partition_id": context.partitionId() if context else None,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "exists": exists,
        "contents": contents,
    }


read_results = sc.parallelize(range(num_tasks), num_tasks).mapPartitions(read_test_file).collect()
pprint(read_results)

## Interpret the shared-file test

The write/read pair checks whether a file created by one task can be read by tasks running elsewhere. If only the writer, or tasks on the writer's host, can see the file, the path is local to that executor host or container. If every task can see the same contents, the path may be suitable as shared storage.

For production-style workflows, prefer object storage, Spark-supported distributed filesystems, or a deliberately mounted shared filesystem over accidental local paths. The important rule is that both the Spark driver and executors must be able to resolve the same path or URL.